In [0]:
data = [
    (1, "2024-01-05", "Tanisha", "Books", 2, 450),
    (2, "2024-01-06", "Rohit", "Electronics", 1, 1200),
    (3, "2024-01-07", "Anjali", "Books", 3, 300),
    (4, "2024-01-08", "Vikas", "Clothing", 5, 700),
    (5, "2024-01-09", "Tanisha", "Books", 1, 550),
    (6, "2024-01-10", None, "Electronics", 2, 2000),
    (7, "2024-01-11", "Amit", "Clothing", 2, 800),
    (8, "2024-01-12", "Neha", "Books", 4, 250),
    (9, "2024-01-13", "Ravi", "Electronics", 1, 1500),
    (10,"2024-01-14", "Pooja", "Books", 2, 600)
]

columns = [
    "order_id",
    "order_date",
    "customer_name",
    "category",
    "quantity",
    "price"
]

df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()


In [0]:
from pyspark.sql.functions import sum,avg,count
df.groupby("category") \
    .agg(
    sum("price").alias("total_Sales"),
    sum("price").alias("avg_price"),
    sum("price").alias("order_count")
    ) \
    .show()

In [0]:
from pyspark.sql.functions import col
df.filter(col("price")>0) \
    .select("category","price")\
    .groupBy("category")\
    .agg(
        sum("price").alias("total_Sum")
    ).show()

In [0]:
from pyspark.sql.functions import count
df.groupBy("category") \
    .agg(count("*").alias("orders")) \
    .show()

In [0]:
from pyspark.sql.functions import avg
df.groupby("category") \
    .agg(avg("price").alias("avg_price")) \
    .show()

In [0]:
from pyspark.sql.functions import to_date,col
df=df.withColumn(
    "order_date_clean",
    to_date(col("order_date"),"yyyy-MM-dd")
)
df.show()

In [0]:
from pyspark.sql.functions import sum

df.groupBy("category", "order_date_clean") \
  .agg(sum("price").alias("daily_sales")) \
  .show()


In [0]:
from pyspark.sql.functions import year , month
df=df.withColumn("order_year",year(col("order_date_clean"))) \
    .withColumn("order_month",month(col("order_date_clean")))
df.show()

In [0]:
df.groupby("category","order_date","order_month") \
    .agg(sum("price").alias("monthly_sales")) \
    .show()

In [0]:
from pyspark.sql.functions import sum,avg,count
df.groupBy("category")\
    .agg(
        sum("price").alias("total_price"),
        avg("price").alias("avg_price"),
        count("*").alias("order_count") 
    )\
        .show()
    

In [0]:
from pyspark.sql.functions import when 
df.groupBy("category") \
    .agg(
        sum(
            when(col("price") >1000 ,col("price"))
        ).alias("High_value_Sales")
    ) \
        .show()

In [0]:
from pyspark.sql.functions  import col
df = df.withColumn(
    "total_amount",
    col("price") * col("quantity")
)

df.show()

In [0]:
print(col)

In [0]:
df.groupBy("category") \
    .agg(sum("total_amount").alias("total_Sales")) \
     .filter(col("total_sales") > 2000) \
    .show()   


In [0]:
df.filter(col("price") > 0) \
    .select("category","order_date_clean","total_amount") \
        .groupBy("category","order_date_clean") \
        .agg(sum("total_amount").alias("sales")).show()

In [0]:
df.groupBy("category","order_date_clean") \
    .agg(sum("total_amount").alias("daily_Sales")) \
    .show()

In [0]:
df.groupBy("category") \
    .agg(
        sum("total_amount").alias("revenue"),
        avg("price").alias("avg_price"),
        count("*").alias("avg_price")
    ) \
    .show()

In [0]:
df.groupBy("category") \
    .agg(count(when(col("price")> 1000 , True)) .alias("high_value_orders")) \
        .show()

In [0]:
df.groupBy("category","order_year","order_month") \
    .agg(sum("total_amount").alias("monthly_Sales")) \
        .show()

# Window funcion 

In [0]:
from pyspark.sql.window import Window

windowSpec = Window \
    .partitionBy("column") \
    .orderBy("column") \
    .rowsBetween(1,5) \
    


In [0]:
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("category")

df = df.withColumn(
    "category_total_sales",
    sum("total_amount").over(windowSpec)
)

df.show()


# Phase 5 table (Orders • Customers • Products • Categories • Payments)

In [0]:
orders_data = [
    (1, 101, 1001, "2024-01-05", 2),
    (2, 102, 1002, "2024-01-06", 1),
    (3, 103, 1001, "2024-01-07", 3),
    (4, 104, 1003, "2024-01-08", 5),
    (5, 101, 1001, "2024-01-09", 1),
    (6, None, 1002, "2024-01-10", 2),   # missing customer
    (7, 105, 1003, "2024-01-11", 2),    # customer not in master
    (8, 102, 1004, "2024-01-12", 4),
    (9, 106, 1002, "2024-01-13", 1),
    (10, 103, 1001, "2024-01-14", 2)
]

orders_cols = [
    "order_id",
    "customer_id",
    "product_id",
    "order_date",
    "quantity"
]

df_orders = spark.createDataFrame(orders_data, orders_cols)
df_orders.show()


In [0]:
customers_data = [
    (101, "Tanisha", "Delhi"),
    (102, "Rohit", "Mumbai"),
    (103, "Anjali", "Pune"),
    (104, "Vikas", "Delhi"),
    (105, "Amit", "Bangalore")
]

customers_cols = [
    "customer_id",
    "customer_name",
    "city"
]

df_customers = spark.createDataFrame(customers_data, customers_cols)
df_customers.show()


In [0]:
products_data = [
    (1001, "Book - Python", 501, 450),
    (1002, "Laptop", 502, 1200),
    (1003, "T-Shirt", 503, 700),
    (1004, "Book - Spark", 501, 550)
]

products_cols = [
    "product_id",
    "product_name",
    "category_id",
    "price"
]

df_products = spark.createDataFrame(products_data, products_cols)
df_products.show()



In [0]:
categories_data = [
    (501, "Books"),
    (502, "Electronics"),
    (503, "Clothing")
]

categories_cols = [
    "category_id",
    "category_name"
]

df_categories = spark.createDataFrame(categories_data, categories_cols)
df_categories.show()


In [0]:
payments_data = [
    (1, "CARD", "SUCCESS"),
    (2, "UPI", "SUCCESS"),
    (3, "CARD", "FAILED"),
    (4, "CASH", "SUCCESS"),
    (5, "CARD", "SUCCESS"),
    (6, "UPI", "FAILED"),
    (7, "CARD", "SUCCESS"),
    (8, "UPI", "SUCCESS")
]

payments_cols = [
    "order_id",
    "payment_mode",
    "payment_status"
]

df_payments = spark.createDataFrame(payments_data, payments_cols)
df_payments.show()


In [0]:
df_oc =df_orders.join(df_customers,"customer_id","left")
df_oc.show()

In [0]:
from pyspark.sql.functions import col
df_oc.filter(col("customer_name").isNull()).show()

In [0]:
df_op = df_orders.join(df_products,"product_id","inner")
df_op.show()

In [0]:
from pyspark.sql.functions import col
df_op = df_op.withColumn(
    "revenue",
    col("quantity")*col("price")
)
df_op.show()

In [0]:
df_opc=df_orders \
    .join(df_products,"product_id","inner") \
    .join(df_categories,"category_id","inner")

df_opc.select("order_id","product_id","category_name","quantity").show()

In [0]:
df_pay = df_orders.join(df_payments,"order_id","left")
df_pay.show()

In [0]:
df_pay.filter(col("payment_status").isNull()).show()

In [0]:
df_pay.filter(col("payment_status") == "FAILED").show()

In [0]:
df_full =df_orders.join(df_payments,"order_id","outer")
df_full.show()

In [0]:
df_clean =df_orders.select("order_id","customer_id","product_id","quantity") \
    .join(df_customers.select("customer_id","customer_name"),"customer_id","left")

df_clean.show()

In [0]:
from pyspark.sql.functions import broadcast
df_bc=df_orders.join(broadcast(df_customers),"customer_id","left")
df_bc.show()

In [0]:
df_fast = df_orders \
    .join(broadcast(df_products), "product_id", "inner") \
    .join(broadcast(df_categories), "category_id", "inner")

df_fast.show()


In [0]:
df_orders.join(df_customers).filter(col("city") == "Delhi").show()

In [0]:
df_orders \
    .filter(col("order_date") >= "2024-01-10") \
    .join(broadcast(df_customers),"customer_id").show()

In [0]:
df_bc.explain()

In [0]:
from pyspark.sql.functions import col
df_fact = df_orders.join(df_payments ,"order_id","left") \
    .filter(col("payment_status") == "success")
df_fact.show()

In [0]:
from pyspark.sql.functions import broadcast

df_enriched = df_fact \
    .join(broadcast(df_customers), "customer_id", "left") \
    .join(broadcast(df_products), "product_id", "left") \
    .join(broadcast(df_categories), "category_id", "left")


In [0]:
df_enriched = df_enriched.withColumn(
    "revenue",
    col("quantity") * col("price")
)

df_enriched.select(
    "order_id", "city", "category_name", "revenue"
).show()


In [0]:
from pyspark.sql.functions import sum

df_category_kpi = df_enriched.groupBy("category_name") \
    .agg(sum("revenue").alias("total_revenue"))

df_category_kpi.show()


# Phase 6 

In [0]:
from pyspark.sql.functions import col
data = [
    (1,"Tanisha","Delhi"),
    (2,"Rohit","Mumbai"),
    (3,"Anjali","Pune")
]
cols=["id","name","city"]
df = spark.createDataFrame(data, cols)
df.show()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("customers_delta")


In [0]:
spark.table("customers_delta").show()


In [0]:
df_read = spark.read \
    .format("delta") \
    .load("customer_delta")
df_read.show()

In [0]:
data = [(1,"A"),(2,"B")]
df=spark.createDataFrame(data,["id","name"])

In [0]:
df.write.format("delta").saveAsTable("demo_table")

In [0]:
spark.table("demo_table")

In [0]:
orders_data = [
    (1, "2024-01-01", "Books", 2, 300),
    (2, "2024-01-02", "Electronics", 1, 1200),
    (3, "2024-01-02", "Books", 3, 300),
    (4, None, "Clothing", 2, 700),        # bad record
    (5, "2024-01-03", "Books", 1, 300)
]

cols = ["order_id", "order_date", "category", "quantity", "price"]

df_raw = spark.createDataFrame(orders_data, cols)
df_raw.show()


In [0]:
# Bronze Layer -Raw ingest (No logic)
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("orders_bronze")

In [0]:
spark.table("orders_bronze").show()